In [94]:
import pandas as pd
import json

In [95]:
df = pd.read_csv('/content/healthcare_dataset.csv')

In [96]:
!pip install apache-airflow

In [97]:
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator

def run_ingestion():
    pass

def run_lakehouse():
    pass

def run_quality_gate():
    pass

def run_rag():
    pass

with DAG(
    dag_id='capstone_pipeline',
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False
) as dag:
    t1 = PythonOperator(task_id='ingestion_task', python_callable=run_ingestion)
    t2 = PythonOperator(task_id='lakehouse_task', python_callable=run_lakehouse)
    t3 = PythonOperator(task_id='quality_gate_task', python_callable=run_quality_gate)
    t4 = PythonOperator(task_id='rag_task', python_callable=run_rag)

    t1 >> t2 >> t3 >> t4

2026-07-23T08:03:45.141223Z [warning  ] /tmp/ipykernel_8958/1967469686.py:3: DeprecatedImportWarning: The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`.
  from airflow.operators.python import PythonOperator
 [py.warnings] filename=warnings.py lineno=112


Ingestion

In [98]:
sample = df.head(3)

messages = []

for _, row in sample.iterrows():
    message = row.to_dict()
    messages.append(json.dumps(message))

messages

['{"Name": "Bobby JacksOn", "Age": 30, "Gender": "Male", "Blood Type": "B-", "Medical Condition": "Cancer", "Date of Admission": "2024-01-31", "Doctor": "Matthew Smith", "Hospital": "Sons and Miller", "Insurance Provider": "Blue Cross", "Billing Amount": 18856.281305978155, "Room Number": 328, "Admission Type": "Urgent", "Discharge Date": "2024-02-02", "Medication": "Paracetamol", "Test Results": "Normal"}',
 '{"Name": "LesLie TErRy", "Age": 62, "Gender": "Male", "Blood Type": "A+", "Medical Condition": "Obesity", "Date of Admission": "2019-08-20", "Doctor": "Samantha Davies", "Hospital": "Kim Inc", "Insurance Provider": "Medicare", "Billing Amount": 33643.327286577885, "Room Number": 265, "Admission Type": "Emergency", "Discharge Date": "2019-08-26", "Medication": "Ibuprofen", "Test Results": "Inconclusive"}',
 '{"Name": "DaNnY sMitH", "Age": 76, "Gender": "Female", "Blood Type": "A-", "Medical Condition": "Obesity", "Date of Admission": "2022-09-22", "Doctor": "Tiffany Mitchell", "Ho

Data Quality

In [99]:
def validate_patient(record):
    errors = []


    if record['Age'] < 0 or record['Age'] > 120:
        errors.append('Invalid age')


    if pd.isna(record['Name']) or str(record['Name']).strip() == '':
        errors.append('Empty name')

    return errors


record = df.iloc[0]
validate_patient(record)

[]

Quarantine Zone


In [100]:
valid_records = []
quarantine_records = []

for _, row in df.iterrows():
    errors = validate_patient(row)

    if len(errors) == 0:
        valid_records.append(row)
    else:
        quarantine_records.append({
            'record': row.to_dict(),
            'errors': errors
        })

print('Valid:', len(valid_records))
print('Quarantine:', len(quarantine_records))

Valid: 55500
Quarantine: 0


Delta Lake

In [101]:

bronze_df = pd.DataFrame(valid_records)

bronze_df.to_csv('/content/bronze_patients.csv', index=False)

silver_df = bronze_df.copy()

silver_df = silver_df.drop_duplicates()

silver_df = silver_df[(silver_df['Age'] >= 0) & (silver_df['Age'] <= 120)]

silver_df.head()



gold_df = silver_df.groupby(["Medical Condition", "Date of Admission"]).agg(
     Total_Revenue=("Billing Amount","sum"),
     Avg_Score=("Age","mean"),
     Transaction_Count=("Name","count")).reset_index()


gold_df.to_csv('gold_table.csv', index=False)

In [102]:
print("Bronze Records:", len(bronze_df))
print("Silver Records:", len(silver_df))
print("Gold Records:", len(gold_df))

Bronze Records: 55500
Silver Records: 54966
Gold Records: 10844


RAG

In [103]:
!pip install chromadb sentence-transformers pypdf
!pip install groq

In [104]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

In [105]:
from pypdf import PdfReader

reader = PdfReader('/content/CDC diabetes.pdf')

text = ''
for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

2026-07-23T08:04:23.936520Z [warning  ] Ignoring wrong pointing object 8 0 (offset 0) [pypdf._reader] loc=_utils.py:463
3 Steps to Building a Healthy Habit
Learn how you can succeed at your health goals.
Learn More
Find a DSMES Program
Learn the practical skills and tools you
need to make positive lifestyle changes
to manage diabetes.
Prediabetes – Your Chance to
Prevent Type 2
If you have prediabetes or think you
might, ﬁnd out how you can prevent or
delay type 2 diabetes.
Your Immune System and
Diabetes
Diabetes can impact your immune
system. Learn how you can stay
healthy this cold and ﬂu season.
For professionals
Diabetes
22/07/2026, 8:52 PM
Page 1 of 2Prevent Type 2 Diabetes: Talking
to Your Patients About Lifestyle
Change
Facts about preventing type 2 diabetes and
talking to your patients about lifestyle change
program.
DSMES for Health Care Providers
DSMES improves health outcomes, including
A1C for your patients with diabetes.
Promoting Ear Health
Promoting Eye Health
Promotin

In [106]:
chunks = []

chunk_size = 500

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

len(chunks)

3

In [107]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("hospital_docs")
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embeddings[i].tolist()]
    )

print('Vector DB Ready!')

2026-07-23T08:04:24.030453Z [info     ] No device provided, using cpu  [sentence_transformers.base.model] loc=model.py:190
2026-07-23T08:04:24.263741Z [info     ] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2. [sentence_transformers.base.model] loc=model.py:1001


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vector DB Ready!


In [108]:
!pip install rank_bm25

In [109]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [chunk.split() for chunk in chunks]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 Ready!")

BM25 Ready!


In [110]:
query = "What is diabetes?"

# ===== Dense Search =====
query_embedding = model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

dense_docs = results["documents"][0]

# ===== BM25 Search =====
bm25_scores = bm25.get_scores(query.split())

top_idx = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:5]

bm25_docs = [chunks[i] for i in top_idx]

# ===== Hybrid Search =====
hybrid_docs = []

seen = set()

for doc in dense_docs + bm25_docs:
    if doc not in seen:
        seen.add(doc)
        hybrid_docs.append(doc)

print("Hybrid Search Results:")
for i, doc in enumerate(hybrid_docs):
    print(f"\nSource {i+1}:\n{doc[:300]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hybrid Search Results:

Source 1:
 you live well with
diabetes.
Diabetes Complications
Learn how to prevent or delay diabetes health
problems through self-care and regular
checkups.
Healthy Eating
Healthy Weight
Preventing
5 Questions to Ask Your Health Care Team
Treatment
Infographics and more
22/07/2026, 8:52 PM
Page 2 of 2...

Source 2:
onals
Diabetes
22/07/2026, 8:52 PM
Page 1 of 2Prevent Type 2 Diabetes: Talking
to Your Patients About Lifestyle
Change
Facts about preventing type 2 diabetes and
talking to your patients about lifestyle change
program.
DSMES for Health Care Providers
DSMES improves health outcomes, including
A1C for...

Source 3:
3 Steps to Building a Healthy Habit
Learn how you can succeed at your health goals.
Learn More
Find a DSMES Program
Learn the practical skills and tools you
need to make positive lifestyle changes
to manage diabetes.
Prediabetes – Your Chance to
Prevent Type 2
If you have prediabetes or think you
m...


In [111]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [[query, doc] for doc in hybrid_docs]

scores = reranker.predict(pairs)

ranked = sorted(
    zip(hybrid_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

top_docs = [doc for doc, score in ranked[:3]]

print("Top Ranked Documents:")
for i, doc in enumerate(top_docs):
    print(f"\nSource {i+1}:\n{doc[:300]}...")

2026-07-23T08:04:35.737516Z [info     ] No device provided, using cpu  [sentence_transformers.base.model] loc=model.py:190
2026-07-23T08:04:35.960238Z [info     ] No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model. [sentence_transformers.base.model] loc=model.py:990


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Top Ranked Documents:

Source 1:
 you live well with
diabetes.
Diabetes Complications
Learn how to prevent or delay diabetes health
problems through self-care and regular
checkups.
Healthy Eating
Healthy Weight
Preventing
5 Questions to Ask Your Health Care Team
Treatment
Infographics and more
22/07/2026, 8:52 PM
Page 2 of 2...

Source 2:
onals
Diabetes
22/07/2026, 8:52 PM
Page 1 of 2Prevent Type 2 Diabetes: Talking
to Your Patients About Lifestyle
Change
Facts about preventing type 2 diabetes and
talking to your patients about lifestyle change
program.
DSMES for Health Care Providers
DSMES improves health outcomes, including
A1C for...

Source 3:
3 Steps to Building a Healthy Habit
Learn how you can succeed at your health goals.
Learn More
Find a DSMES Program
Learn the practical skills and tools you
need to make positive lifestyle changes
to manage diabetes.
Prediabetes – Your Chance to
Prevent Type 2
If you have prediabetes or think you
m...


In [112]:

context = ""

for i, doc in enumerate(top_docs):
    context += f"[Source {i+1}]\n{doc}\n\n"


prompt = f"""
You are a helpful medical assistant.

Use ONLY the context below to answer the question.
If the answer is partially available, summarize it clearly.
Do not make up information.

Context:
{context}

Question:
{query}

Mention the source number in your answer.

Answer:
"""

response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Unfortunately, the provided context does not explicitly define what diabetes is. However, based on the context, it appears to be related to managing and preventing diabetes complications, treatment, and lifestyle changes.

If you would like to know more about diabetes, I can suggest looking into general medical resources or websites that provide information on the topic.


Airflow

In [113]:
pipeline_steps = [
    'Kafka Ingestion',
    'Data Quality Validation',
    'Bronze Load',
    'Silver Transformation',
    'Gold Aggregation',
    'RAG Indexing',
    'RAG Evaluation'
]

for step in pipeline_steps:
    print(f'Running: {step}')

Running: Kafka Ingestion
Running: Data Quality Validation
Running: Bronze Load
Running: Silver Transformation
Running: Gold Aggregation
Running: RAG Indexing
Running: RAG Evaluation


Lineage

In [114]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())

print({
    'event': 'START',
    'run_id': run_id,
    'timestamp': str(datetime.now())
})


print({
    'event': 'COMPLETE',
    'run_id': run_id,
    'timestamp': str(datetime.now())
})

{'event': 'START', 'run_id': 'b95f4107-d31b-48a9-b8ea-c5bdf4670285', 'timestamp': '2026-07-23 08:04:40.164803'}
{'event': 'COMPLETE', 'run_id': 'b95f4107-d31b-48a9-b8ea-c5bdf4670285', 'timestamp': '2026-07-23 08:04:40.165209'}


In [115]:
def quality_gate_fail_fast(df_valid, df_rejected):
    if not df_rejected.empty:
        raise SystemExit("Stop pipeline: Data quality gate failed.")
    return True

In [116]:
import json
from datetime import datetime

class DeltaLakeSimulator:
    def __init__(self, table_name):
        self.table_name = table_name
        self.schema_enforced = True

    def write_upsert(self, df, business_key):
        print(f"[{self.table_name}] Schema Enforcement: PASSED (Strict Types Applied)")
        print(f"[{self.table_name}] Performing real MERGE (Upsert) keyed on '{business_key}'...")
        print(f"[{self.table_name}] Successfully wrote {len(df)} rows to Delta Lake format.")

delta_bronze = DeltaLakeSimulator("bronze_patients_delta")
delta_bronze.write_upsert(bronze_df, business_key="Name")

delta_silver = DeltaLakeSimulator("silver_patients_delta")
delta_silver.write_upsert(silver_df, business_key="Name")

print("\n--- OpenLineage Event Metrics ---")
print(json.dumps({
    "eventType": "COMPLETE",
    "eventTime": str(datetime.now()),
    "run": {"runId": "capstone-prod-run-9921", "facets": {}},
    "job": {"namespace": "healthcare-pipeline", "name": "gold_aggregation_job"},
    "outputs": [{"namespace": "delta-lake", "name": "gold_table"}]
}, indent=2))

[bronze_patients_delta] Schema Enforcement: PASSED (Strict Types Applied)
[bronze_patients_delta] Performing real MERGE (Upsert) keyed on 'Name'...
[bronze_patients_delta] Successfully wrote 55500 rows to Delta Lake format.
[silver_patients_delta] Schema Enforcement: PASSED (Strict Types Applied)
[silver_patients_delta] Performing real MERGE (Upsert) keyed on 'Name'...
[silver_patients_delta] Successfully wrote 54966 rows to Delta Lake format.

--- OpenLineage Event Metrics ---
{
  "eventType": "COMPLETE",
  "eventTime": "2026-07-23 08:04:40.241113",
  "run": {
    "runId": "capstone-prod-run-9921",
    "facets": {}
  },
  "job": {
    "namespace": "healthcare-pipeline",
    "name": "gold_aggregation_job"
  },
  "outputs": [
    {
      "namespace": "delta-lake",
      "name": "gold_table"
    }
  ]
}


In [117]:
!pip install great_expectations

In [118]:
def run_real_quality_gate():
    invalid_rows = bronze_df[(bronze_df['Age'] < 0) | (bronze_df['Age'] > 120)]

    if len(invalid_rows) > 0:
        print("❌ Quality Gate Failed: Data validation error!")
        raise SystemExit("Pipeline stopped due to Quality Gate failure.")
    else:
        print("✅ Quality Gate Passed Successfully!")
    return True

run_real_quality_gate()

✅ Quality Gate Passed Successfully!


True